## Train 62-class recognizer on the app input distribution (GPU)

Automated run for `pi-learn-station`. Everything below is hands-off.
This notebook must run on a **GPU** runtime.

### How to run
1. **Runtime → Change runtime type → Hardware accelerator → GPU → Save**.
2. **Runtime → Run all** (or press `Ctrl+F9`).
3. Watch for `===TRAINING_DONE===` in the final cell output.
4. The three files (`model.onnx`, `config.json`, `model.pt`) download to your browser's Downloads folder automatically.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'opencv-python-headless', 'onnxruntime', 'scipy'],
               check=True)
import numpy, cv2, onnxruntime
print('deps ok', numpy.__version__, cv2.__version__)


In [ ]:
# Download the exact raw EMNIST ByClass files used locally (HF mirror).
import gzip, os, urllib.request
BASE = ('https://huggingface.co/datasets/Royc30ne/emnist-byclass/'
        'resolve/main/emnist-byclass-')
FILES = ['train-images-idx3-ubyte.gz', 'train-labels-idx1-ubyte.gz',
         'test-images-idx3-ubyte.gz', 'test-labels-idx1-ubyte.gz']
RAW = '/content/emnist_raw'
os.makedirs(RAW, exist_ok=True)
for name in FILES:
    local = os.path.join(RAW, name)
    if os.path.exists(local) and os.path.getsize(local) > 0:
        print('cached', name); continue
    url = BASE + name + '?download=true'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=300) as r, open(local, 'wb') as f:
        f.write(r.read())
    print('downloaded', name, os.path.getsize(local))
print('all raw files ready')


In [ ]:
# Fetch the prep + train scripts from this repo (GitHub main).
import os, urllib.request
os.makedirs('/content/ml', exist_ok=True)
BASE = ('https://raw.githubusercontent.com/sujith0613/pi-learn-station/'
        'main/ml/')
for name in ['prep_emnist_render.py', 'train_recog.py']:
    url = BASE + name
    with urllib.request.urlopen(url, timeout=60) as r:
        src = r.read().decode()
    with open('/content/ml/' + name, 'w') as f:
        f.write(src)
    print('fetched', name, len(src), 'bytes')


In [ ]:
# Build the app-normalized + augmented training set (idempotent).
import os, subprocess, sys, time
os.makedirs('/content/data/processed', exist_ok=True)
OUT = '/content/data/processed/train62_uint8.npz'
if os.path.exists(OUT):
    print('prep cache exists, skipping')
else:
    env = dict(os.environ,
               EMNIST_DATA='/content/emnist_raw',
               EMNIST_OUT='/content/data/processed',
               EMNIST_VARIANTS='3')
    t0 = time.time()
    r = subprocess.run([sys.executable, '/content/ml/prep_emnist_render.py'],
                      env=env)
    print('prep done in', int(time.time()-t0), 's, rc', r.returncode)


In [ ]:
# Train 12 epochs on the GPU runtime and export ONNX.
import os, subprocess, sys, time
env = dict(os.environ,
           EMNIST_PROC='/content/data/processed',
           EMNIST_OUT='/content/models/recog',
           EMNIST_EPOCHS='12',
           EMNIST_BATCH='512')
t0 = time.time()
r = subprocess.run([sys.executable, '/content/ml/train_recog.py'], env=env)
print('train rc', r.returncode, 'in', int(time.time()-t0), 's')


In [ ]:
# Verify the exported ONNX matches torch, then download artifacts.
import os, sys, numpy as np, torch, onnxruntime as ort
sys.path.insert(0, '/content/ml')
import train_recog as T
m = T.ResNet(62)
m.load_state_dict(torch.load('/content/models/recog/model.pt',
                             map_location='cpu'))
m.eval()
sess = ort.InferenceSession('/content/models/recog/model.onnx',
                            providers=['CPUExecutionProvider'])
x = torch.rand(4, 28, 28, 1)
ort_out = sess.run(None, {'input': x.numpy()})[0]
torch_out = m.forward_nhwc(x).detach().numpy()
md = float(np.abs(ort_out - torch_out).max())
print('onnx max diff vs torch:', md)
assert md < 1e-3, 'ONNX mismatch'
print('ONNX verified')
from google.colab import files
for f in ['model.onnx', 'config.json', 'model.pt']:
    p = '/content/models/recog/' + f
    print('downloading', f, os.path.getsize(p))
    files.download(p)
print('===TRAINING_DONE===')
